# Policy-Based Reinforcement Learning

## Policy Function Approximation

- 策略函数$\pi(a|s)$是一个概率密度函数；
- 输入为当前状态`s`；
- 输出为当前`agent`采取各种行为的概率；

只要有一个好的策略函数$\pi(a|s)$，就可以通过其控制`agent`自动采取行为。
- 通过策略网络$\pi(a|s; \theta)$来拟合$\pi(a|s)$；
- $\theta$为神经网络的参数。

策略网络$\pi(a|s; \theta)$满足：$\sum_{a \in \mathcal A} \pi(a|s; \theta) = 1$

$$
U_t = R_t + \gamma R_{t+1} + \gamma ^ 2 R_{t+2} + \gamma ^ 3 R_{t+3} + ...
$$

行为价值函数：
$$
Q_\pi (s_t, a_t) = \mathbb E [U_t | S_t = s_t, A_t = a_t]
$$

状态价值函数：
$$
V_\pi(s_t) = \mathbb E_A [Q_\pi (s_t, A)] = \sum_a \pi(a|s_t) \cdot Q_\pi(s_t, a).
$$
随机变量$A \sim \pi(\cdot | s_t)$.

## Policy-Based Reinforcement Learning

采用策略网络$\pi(a|s_t, \theta)$来拟合策略函数$\pi(a|s_t)$

$$
V(s_t; \theta) = \sum_a \pi(a|s_t; \theta) \cdot Q_\pi(s_t, a).
$$

通过改进网络参数$\theta$，使得$V(s_t; \theta)$变大；

基于以上思想，可以得到目标函数：$J(\theta) = \mathbb E_S[V(S; \theta)]$.

**如何提升**$\theta$**？**

Policy gradient ascent（策略梯度算法）
- 观测到状态`s`；
- 更新策略：$\theta \leftarrow \theta + \beta \cdot \frac {\partial V(s; \theta)}{\partial \theta}$.

## Policy Gradient

近似状态价值函数：
$$
V(s_t; \theta) = \sum_a \pi(a|s_t; \theta) \cdot Q_\pi(s_t, a).
$$
策略梯度：$\frac {\partial V(s; \theta)}{\partial \theta} = \frac {\partial \sum_a \pi(a|s; \theta) \cdot Q_\pi(s, a)}{\partial \theta} = \sum_a \frac {\partial_\pi (a|s; \theta) \cdot Q_\pi(s, a)}{\partial \theta} = \sum_a \frac {\partial \pi(a|s \theta)}{\partial \theta} \cdot Q_\pi(s, a)$

假设$Q_\pi$不依赖$\theta$，其实是依赖的，简化操作。

其中：$\frac {\partial \pi(a|s; \theta)}{\partial \theta} = \pi(a|s; \theta) \cdot \frac {\partial log \pi(a|s; \theta)}{\partial \theta}$，（$\frac {\partial log[\pi(\theta)]}{\partial \theta} = \frac 1 \pi(\theta) \cdot \frac {\partial \pi(\theta)}{\partial \theta}$）
$\rightarrow \pi(\theta) \cdot \frac {\partial log[\pi(\theta)]}{\partial \theta} = \pi(\theta) \cdot \frac 1 \pi(\theta) \cdot \frac {\partial \pi(\theta)}{\partial \theta} = \frac {\partial \pi(\theta)}{\partial \theta}$.

$\frac {\partial V(s; \theta)}{\partial \theta} = \sum_a \frac {\partial \pi(a|s; \theta)}{\partial \theta} \cdot Q_\pi(s, a) = \sum_a \pi(a|s; \theta) \cdot \frac {\partial log \pi(a|s; \theta)}{\partial \theta} \cdot Q_\pi(s, a) = \mathbb E_A[\frac {\partial log \pi(A|s; \theta)}{\partial \theta} \cdot Q_\pi(s, A)]$

对于离散的策略：
$$
f(a, \theta) = \frac {\partial \pi(a|s; \theta)}{\partial \theta} \cdot Q_\pi(s, a), a in \mathcal A
$$
策略梯度：
$$
\frac {\partial V(s; \theta)}{\partial \theta} = f(left, \theta) + f(right, \theta) + f(up, \theta)
$$

对于连续的策略：

$$
\frac {\partial V(s; \theta)}{\partial \theta} = \mathbb E_{A \sim} \pi(\cdot|s; \theta) [\frac {\partial log \pi(A|s, \theta)}{\partial \theta} \cdot Q_\pi(s, A)]
$$

由于$A$为神经网络，无法直接通过公式求出期望，故采用蒙特卡洛方法计算其近似：
- 通过概率密度函数$\pi(\cdot|s; \theta)$抽样出一个策略$\hat a$；
- 计算$g(\hat a, \theta) = \frac {\partial log \pi(\hat a | s; \theta)}{\partial \theta} \cdot Q_\pi(s, \hat a)$.

显然，$\mathbb E_A[g(A, \theta)] = \frac {\partial V(s; \theta)}{\theta}$，$g(\hat a, \theta)$为$\frac {\partial V(s, \theta)}{\partial \theta}$的无偏估计。

- 使用$g(\hat a, \theta)$作为策略梯度$\frac {\partial V(s; \theta)}{\partial \theta}$的近似。

流程：
- 观察状态$s_t$；
- 根据$\pi(\cdot| s_t; \theta_t)$随机抽样行为$a_t$；
- 计算$q_t \approx Q_\pi(s_t, a_t)$；
- 定义策略网络：$d_{\theta, t} = \frac {\partial log \pi (a_t|s_t, \theta)}{\partial \theta} | \theta = \theta_t$；
- 蒙特卡洛方法计算策略梯度：$g(a_t, \theta_t) = q_t \cdot d_{\theta, t}$；
- 更新策略网络：$\theta_{t+1} = \theta_t + \beta \cdot g(a_t, \theta_t)$。